# 00 — California boundary and centre-based spatial grid

Reconstructs the fixed spatial domain used by every model.

- 2024 U.S. Census TIGER/Line state boundary
- bounding box: 32.0–42.5°N, 125.0–114.0°W
- 53 × 55 regular grid
- retain a cell if its centre lies inside California
- expected retained cells: 1,080

In [ ]:
from pathlib import Path
import io, os, sys, zipfile
import numpy as np
import pandas as pd
import requests

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
FIGURE_DIR = ROOT / "outputs" / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT.resolve())

In [ ]:
# ==================================================
# California state boundary: PROJ setup + cached download
# ==================================================

import os
import sys
import io
import zipfile
from pathlib import Path

import requests

# --------------------------------------------------
# 1. Configure PROJ before importing geopandas/pyproj
#    This avoids the common Windows/Anaconda PROJ warning.
# --------------------------------------------------
proj_candidates = [
    Path(sys.prefix) / "Library" / "share" / "proj",
    Path(sys.prefix) / "share" / "proj",
]

proj_data_dir = next(
    (p for p in proj_candidates if (p / "proj.db").exists()),
    None
)

if proj_data_dir is not None:
    os.environ["PROJ_DATA"] = str(proj_data_dir)
    os.environ["PROJ_LIB"] = str(proj_data_dir)
else:
    print("Warning: proj.db was not found in the usual Anaconda locations.")
    print("Checked:")
    for p in proj_candidates:
        print(" -", p)

import pyproj

if proj_data_dir is not None:
    pyproj.datadir.set_data_dir(str(proj_data_dir))

import geopandas as gpd

print("PROJ data directory:", pyproj.datadir.get_data_dir())

# --------------------------------------------------
# 2. Download/cache the US Census state shapefile
# --------------------------------------------------
BOUNDARY_DIR = DATA_DIR / "boundary_data"
extract_dir = BOUNDARY_DIR / "tl_2024_us_state"
shapefile_path = extract_dir / "tl_2024_us_state.shp"

STATE_SHAPEFILE_URL = (
    "https://www2.census.gov/geo/tiger/TIGER2024/STATE/"
    "tl_2024_us_state.zip"
)

if not shapefile_path.exists():
    print("State boundary shapefile not found locally.")
    print("Downloading 2024 Census TIGER/Line state boundaries...")

    extract_dir.mkdir(parents=True, exist_ok=True)

    response = requests.get(STATE_SHAPEFILE_URL, timeout=120)
    response.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
        zf.extractall(extract_dir)

# Fallback in case the internal filename changes
if not shapefile_path.exists():
    shp_candidates = list(extract_dir.glob("*.shp"))
    if not shp_candidates:
        raise FileNotFoundError(
            f"No shapefile found in {extract_dir.resolve()}"
        )
    shapefile_path = shp_candidates[0]

print("Using state boundary file:", shapefile_path.resolve())

# --------------------------------------------------
# 3. Load US states and retain California
# --------------------------------------------------
us_states = gpd.read_file(
    shapefile_path,
    engine="pyogrio"
)

print("Original CRS:", us_states.crs)

california_boundary = us_states[
    us_states["STUSPS"] == "CA"
].copy()

if california_boundary.empty:
    raise ValueError("California (STUSPS='CA') was not found in the state shapefile.")

california_boundary = california_boundary.to_crs("EPSG:4326")

print("California boundary rows:", len(california_boundary))
print("California CRS:", california_boundary.crs)

display(
    california_boundary[
        ["STATEFP", "STUSPS", "NAME", "geometry"]
    ]
)

## Build the regular grid

In [ ]:
import numpy as np
import geopandas as gpd
from shapely.geometry import box

# Study-region boundaries
lat_min, lat_max = 32.0, 42.5
lon_min, lon_max = -125.0, -114.0

# Grid dimensions
n_lat = 53
n_lon = 55

# Grid edges
lat_edges = np.linspace(lat_min, lat_max, n_lat + 1)
lon_edges = np.linspace(lon_min, lon_max, n_lon + 1)

grid_records = []

for lat_idx in range(n_lat):
    for lon_idx in range(n_lon):

        south = lat_edges[lat_idx]
        north = lat_edges[lat_idx + 1]
        west = lon_edges[lon_idx]
        east = lon_edges[lon_idx + 1]

        cell_id = lat_idx * n_lon + lon_idx

        grid_records.append({
            "cell_id": cell_id,
            "lat_idx": lat_idx,
            "lon_idx": lon_idx,
            "south": south,
            "north": north,
            "west": west,
            "east": east,
            "centre_lat": (south + north) / 2,
            "centre_lon": (west + east) / 2,
            "geometry": box(west, south, east, north)
        })

grid_gdf = gpd.GeoDataFrame(
    grid_records,
    geometry="geometry",
    crs="EPSG:4326"
)

print("Total grid cells:", len(grid_gdf))

## Apply the centre-based mask

In [ ]:
grid_centres_gdf = gpd.GeoDataFrame(
    grid_gdf[
        [
            "cell_id",
            "lat_idx",
            "lon_idx",
            "centre_lon",
            "centre_lat"
        ]
    ].copy(),
    geometry=gpd.points_from_xy(
        grid_gdf["centre_lon"],
        grid_gdf["centre_lat"]
    ),
    crs="EPSG:4326"
)

california_grid_centres = gpd.sjoin(
    grid_centres_gdf,
    california_boundary[["geometry"]],
    how="inner",
    predicate="within"
)

california_cell_ids = set(
    california_grid_centres["cell_id"]
)

grid_gdf["is_california"] = (
    grid_gdf["cell_id"].isin(california_cell_ids)
)

california_grid_gdf = grid_gdf[
    grid_gdf["is_california"]
].copy()

print("Total bounding-box cells:", len(grid_gdf))
print("California mask cells:", len(california_grid_gdf))

print("Total bounding-box cells:", len(grid_gdf))
print("California mask cells:", len(california_grid_gdf))
print(
    "Mask proportion:",
    round(len(california_grid_gdf) / len(grid_gdf), 4)
)

In [ ]:
assert len(grid_gdf) == 2915
assert len(california_grid_gdf) == 1080
assert california_grid_gdf["cell_id"].is_unique
assert california_grid_gdf["is_california"].all()

mask_csv = DATA_DIR / "california_grid_centre_mask.csv"
mask_geojson = DATA_DIR / "california_grid_centre_mask.geojson"

california_grid_gdf.drop(columns="geometry").to_csv(mask_csv, index=False)
california_grid_gdf.to_file(mask_geojson, driver="GeoJSON")

print("Saved:", mask_csv)
print("Saved:", mask_geojson)

## Spatial-domain figures

In [ ]:
import matplotlib.pyplot as plt
from shapely.geometry import box

bbox_geom = box(lon_min, lat_min, lon_max, lat_max)

fig, ax = plt.subplots(figsize=(7, 8))
california_boundary.plot(ax=ax, facecolor="0.92", edgecolor="black", linewidth=1.0)
gpd.GeoSeries([bbox_geom], crs="EPSG:4326").boundary.plot(
    ax=ax, edgecolor="black", linewidth=1.5, linestyle="--"
)
ax.set_xlim(lon_min - 0.5, lon_max + 0.5)
ax.set_ylim(lat_min - 0.5, lat_max + 0.5)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("California catalogue-retrieval bounding box")
plt.tight_layout()
fig.savefig(FIGURE_DIR / "california_bounding_box.png", dpi=300, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 10))
california_grid_gdf.boundary.plot(ax=ax, linewidth=0.25)
california_boundary.boundary.plot(ax=ax, edgecolor="black", linewidth=1.2)
ax.set_xlim(lon_min, lon_max)
ax.set_ylim(lat_min, lat_max)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Centre-based California grid mask")
plt.tight_layout()
fig.savefig(FIGURE_DIR / "california_grid_mask_centre_based.png", dpi=300, bbox_inches="tight")
plt.show()